# Numerical identities and failure handling

Check the numerical identities used by geometric information decomposition (GID), including moment matching, nested log-score gaps, basis invariance, and boundary diagnostics. Failure-injection checks verify that unacceptable refined fits make inference unavailable.

Both modes run the same numerical checks. These checks assess implementation behavior; they do not establish uniform numerical error bounds or finite-sample statistical coverage.

The notebook calls the shared workflow and reads its saved results. It does not implement a second fitting routine. Generated outputs go to `results/smoke` or `results/paper`; the frozen `reference_results` directory is not overwritten.

Run all cells from top to bottom after changing the mode. Smoke mode checks execution and output structure. Its tiny simulation samples are not evidence for the manuscript's statistical conclusions.


In [1]:
# Change MODE to "paper" to run the complete manuscript experiment.
# Publication figures require LaTeX and the configured image-conversion tools.
MODE = "smoke"
FIGURES = False
assert MODE in {"smoke", "paper"}


In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
REPO = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "workflow.py").is_file() and (p / "code" / "gid_pipeline.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Open this notebook from the repository root or its notebooks directory.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from workflow import run_study


def read_json(path):
    return json.loads(Path(path).read_text())


def require_finite(frame, columns):
    values = frame.loc[:, columns].apply(pd.to_numeric, errors="raise").to_numpy()
    assert np.isfinite(values).all(), f"Nonfinite values in {columns}"


def show_run_figure(relative_path):
    if not FIGURES:
        print("Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.")
        return
    figure = RUN / relative_path
    if not figure.is_file():
        raise FileNotFoundError(f"The requested current-run figure was not generated: {figure}")
    display(Markdown(f"Figure from `{figure.relative_to(REPO)}` ({MODE} mode)."))
    display(Image(filename=str(figure)))

print(f"Repository root located: {REPO.name}")
print(f"Mode: {MODE}; figures: {FIGURES}")


Repository root located: github
Mode: smoke; figures: False


## Run the checks

`run_study("checks", ...)` uses the common fitting engine and writes the check reports. This notebook does not need figures, so `FIGURES` has no display role here.


In [3]:
RUN = Path(run_study("checks", mode=MODE, figures=FIGURES)).resolve()
assert RUN == (REPO / "results" / MODE).resolve()
assert RUN != (REPO / "reference_results").resolve()
print(f"Reading generated results from {RUN.relative_to(REPO)}")


smoke: numerical_checks


smoke: refinement_gate_checks


Reading generated results from results/smoke


In [4]:
unit = read_json(RUN / "study_b" / "numerical_unit_checks.json")
gates = read_json(RUN / "study_b" / "refinement_gate_checks.json")
unit_table = pd.DataFrame(unit["checks"])
gate_table = pd.DataFrame(gates["checks"])
display(unit_table)
display(gate_table[["name", "expected", "actual", "passed"]])

assert unit["total"] == 32 == len(unit_table)
assert unit["passed"] == unit["total"]
assert unit_table["passed"].all()
require_finite(unit_table, ["error", "tolerance"])
assert (unit_table["error"] <= unit_table["tolerance"]).all()
assert gates["total"] == 9 == len(gate_table)
assert gates["passed"] and gate_table["passed"].all()
assert (gate_table["expected"] == gate_table["actual"]).all()
print(f"Passed {len(unit_table)} numerical checks and {len(gate_table)} refinement-status checks.")


,check,error,tolerance,passed
0,uniform_degree1,0.000000e+00,1.000000e-10,True
1,uniform_degree2,0.000000e+00,1.000000e-10,True
2,uniform_degree3,0.000000e+00,1.000000e-10,True
3,uniform_degree4,0.000000e+00,1.000000e-10,True
4,vm_later_gaps,8.881784e-16,1.000000e-10,True
5,cos4_first_three_gaps,0.000000e+00,1.000000e-10,True
6,cos4_fourth_positive,0.000000e+00,0.000000e+00,True
7,cos4_reference_eigenvalues,1.776357e-15,1.000000e-10,True
8,local_first_order_d2,6.077944e-09,1.000000e-05,True
9,local_mean_zero_quadratic_d2,1.216143e-08,1.000000e-05,True


,name,expected,actual,passed
0,unchecked_preserves_outputs,not_checked,not_checked,True
1,successful_refinement_preserves_outputs,ok,ok,True
2,failed_full_fit_gates_every_inference_output,unavailable_refined_fit,unavailable_refined_fit,True
3,failed_reduced_fit_gates_every_inference_output,unavailable_refined_fit,unavailable_refined_fit,True
4,missing_required_reduced_fit_unavailable,unavailable_refined_fit,unavailable_refined_fit,True
5,nonfinite_refined_gap_unavailable,unavailable_refined_fit,unavailable_refined_fit,True
6,nonfinite_refined_parameter_unavailable,unavailable_refined_fit,unavailable_refined_fit,True
7,excessive_scaled_difference_gates_every_infere...,excessive_scaled_difference,excessive_scaled_difference,True
8,subthreshold_scaled_difference_preserves_outputs,ok,ok,True


Passed 32 numerical checks and 9 refinement-status checks.


## Interpret the reports

A small fitting residual supports the recorded fit. It is not a certificate for every possible dataset or quadrature rule. A refinement marked `not_checked` differs from a successful refinement check.

The refinement tolerance is an operational diagnostic introduced during the implementation audit. Passing it does not establish the asymptotic numerical-error conditions in the theory. The report also retains boundary and grid-infeasibility outcomes.


In [5]:
display(pd.DataFrame({
    "diagnostic": ["boundary fit status", "grid target feasible", "refinement tolerance"],
    "value": [unit["boundary_status"], unit["grid_feasibility"]["feasible"], gates["tolerance"]],
}))
print(unit["note"])
print(gates["provenance"])


,diagnostic,value
0,boundary fit status,boundary_or_ill_conditioned
1,grid target feasible,False
2,refinement tolerance,0.001


Numerical checks diagnose implementation; they do not certify asymptotic error rates or uniform coverage.
Failure injection unit checks using a fixed small fitted fixture, not an additional calibration experiment.
